# Silver Layer - Grid Events Transformation

**Purpose**: Transform bronze_grid_events into silver_grid_events with:
- Timestamp parsing and event_day field
- Severity normalization (critical/high/medium/low)
- Duration cleaning and validation
- Event-type normalization
- Severity band classifier UDF
- Electrical grid calculations using Grid UDFs

**Input**: `vattenfall_dev.raw.bronze_grid_events`

**Output**: `vattenfall_dev.refined.silver_grid_events`

**Key Transformations**:
1. Normalize event types and severity levels
2. Create event_day field from timestamp
3. Filter invalid durations and affected customers
4. Add severity band classifications
5. Calculate event statistics and operational context
6. Enrich with metadata

In [0]:
# Setup Python path to access project modules
import sys
project_root = "/Workspace/Users/gharbi@startsteps.org/vattenfall-capstone-project"
if project_root not in sys.path:
    sys.path.append(project_root)

# Import transformation modules
from src.transforms import grid_event_transforms
from src.udfs import grid_udfs
from src.validation.silver_checks import validate_grid_events
import importlib

# Reload modules to pick up latest changes
importlib.reload(grid_event_transforms)
importlib.reload(grid_udfs)

# Register Grid UDFs for electrical calculations
grid_udfs.register_udfs(spark)

print("✓ Setup complete")
print(f"✓ Grid UDFs registered")
print(f"✓ Project root: {project_root}")
print(f"✓ Modules loaded: grid_event_transforms, grid_udfs, silver_checks")

In [0]:
# Read bronze grid events data
df_bronze = spark.table("vattenfall_dev.raw.bronze_grid_events")

print(f"Bronze Grid Events Loaded:")
print(f"  Records: {df_bronze.count():,}")
print(f"  Columns: {len(df_bronze.columns)}")
print(f"\nSchema:")
df_bronze.printSchema()

# Show sample records
print("\nSample Records:")
display(df_bronze.limit(5))

In [0]:
# Analyze event types and severity distributions
from pyspark.sql import functions as F

print("=== Event Type Distribution ===")
event_type_dist = df_bronze.groupBy("event_type").count().orderBy(F.col("count").desc())
display(event_type_dist)

print("\n=== Severity Distribution ===")
severity_dist = df_bronze.groupBy("severity").count().orderBy(F.col("count").desc())
display(severity_dist)

print("\n=== Event Type + Severity Cross-Tabulation ===")
event_severity = df_bronze.groupBy("event_type", "severity").count().orderBy("event_type", F.col("count").desc())
display(event_severity)

print("\n=== Duration and Affected Customers Statistics ===")
stats = df_bronze.select(
    F.min("duration_minutes").alias("min_duration"),
    F.max("duration_minutes").alias("max_duration"),
    F.avg("duration_minutes").alias("avg_duration"),
    F.min("affected_customers").alias("min_affected"),
    F.max("affected_customers").alias("max_affected"),
    F.avg("affected_customers").alias("avg_affected"),
    F.count("*").alias("total_records")
)
display(stats)

In [0]:
# Apply the complete transformation pipeline
df_silver = grid_event_transforms.transform_bronze_to_silver(df_bronze)

print(f"Silver Grid Events Transformed:")
print(f"  Records: {df_silver.count():,}")
print(f"  Columns: {len(df_silver.columns)}")
print(f"\nNew Columns Added:")
bronze_cols = set(df_bronze.columns)
silver_cols = set(df_silver.columns)
new_cols = sorted(silver_cols - bronze_cols)
for col in new_cols:
    print(f"  - {col}")

print(f"\nTotal new columns: {len(new_cols)}")

In [0]:
# Display sample of transformed silver data
print("=== Silver Grid Events Sample ===")
display(df_silver.limit(10))

print("\n=== Key Transformations Verification ===")
print("\n1. Event Day Field:")
display(df_silver.select("timestamp", "event_day").limit(5))

print("\n2. Normalized Event Types:")
display(df_silver.groupBy("event_type_normalized").count().orderBy("event_type_normalized"))

print("\n3. Normalized Severity Levels:")
display(df_silver.groupBy("severity_normalized").count().orderBy("severity_normalized"))

print("\n4. Severity Bands (from UDF):")
display(df_silver.groupBy("severity_band").count().orderBy("severity_band"))

In [0]:
# Write the silver dataframe to the refined schema
table_name = "vattenfall_dev.refined.silver_grid_events"

df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Silver table written successfully: {table_name}")

# Verify the written table
df_verify = spark.table(table_name)
print(f"\nVerification:")
print(f"  Records in table: {df_verify.count():,}")
print(f"  Columns in table: {len(df_verify.columns)}")

In [0]:
# Run validation checks on the silver table
df_silver_table = spark.table("vattenfall_dev.refined.silver_grid_events")

print("=== Running Silver Data Quality Validation ===")
print()

validation_results = validate_grid_events(df_silver_table)

print(validation_results)

In [0]:
# Final summary of the silver grid events transformation
df_bronze = spark.table("vattenfall_dev.raw.bronze_grid_events")
df_silver = spark.table("vattenfall_dev.refined.silver_grid_events")

bronze_count = df_bronze.count()
silver_count = df_silver.count()
bronze_cols = len(df_bronze.columns)
silver_cols = len(df_silver.columns)
filtered_count = bronze_count - silver_count

print("="*60)
print("SILVER GRID EVENTS TRANSFORMATION SUMMARY")
print("="*60)
print(f"\n📊 Record Counts:")
print(f"   Bronze Records:  {bronze_count:>6,}")
print(f"   Silver Records:  {silver_count:>6,}")
print(f"   Filtered Out:    {filtered_count:>6,} ({filtered_count/bronze_count*100:.1f}%)")
print(f"\n📋 Column Counts:")
print(f"   Bronze Columns:  {bronze_cols:>6}")
print(f"   Silver Columns:  {silver_cols:>6}")
print(f"   Columns Added:   {silver_cols - bronze_cols:>6}")
print(f"\n✅ Key Transformations:")
print(f"   • Timestamp parsing and event_day field created")
print(f"   • Event types normalized (case-insensitive)")
print(f"   • Severity levels normalized to 4 levels")
print(f"   • Duration and affected customers validated")
print(f"   • Severity band classifier UDF applied")
print(f"   • Event statistics and operational context added")
print(f"   • Grid UDFs available for electrical calculations")
print(f"\n📍 Table Location: vattenfall_dev.refined.silver_grid_events")
print("="*60)